In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
save_dir = "/content/drive/MyDrive/NLP_Project_Preprocessing"

In [3]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

In [4]:
!pip install -q tf-keras

In [ ]:
import os

os.makedirs(save_dir, exist_ok=True)
print(os.listdir(save_dir))

['sentiment_encoder.pkl', 'stance_encoder.pkl', 'train_idx.npy', 'test_idx.npy', 'y_train_stance.npy', 'y_test_stance.npy', 'y_train_sentiment.npy', 'y_test_sentiment.npy', 'X_train.csv', 'X_test.csv', 'X_train_tokens.pkl', 'X_test_tokens.pkl', 'word_index.pkl', 'word2vec.model', 'X_train_pad.npy', 'X_test_pad.npy', 'embedding_matrix.npy', 'rnn_stance.keras', 'rnn_sentiment.keras', 'test_original_tweets.csv', 'train_original_tweets.csv', 'predictions.csv', 'bert_stance', 'lstm_stance.keras', 'config.json', 'tf_model.h5', 'tokenizer_config.json', 'special_tokens_map.json', 'vocab.txt', 'tokenizer.json']


In [5]:
import pandas as pd

X_train = pd.read_csv(f"{save_dir}/X_train.csv")
X_test = pd.read_csv(f"{save_dir}/X_test.csv")

In [ ]:
print(X_train.shape)
print(X_train.head())
print(X_train.columns)

(1166475, 1)
                                  reconstructed_text
0       ukraine rejects russian neutrality proposals
1  citizens protesting zahedan injured clashes re...
2    pope francis send emissaries russia ukraine war
3  new spoke parents prosper adopted orphan broth...
4  live one month start war ukraine pan american ...
Index(['reconstructed_text'], dtype='object')


In [ ]:
print(len(X_train))
print(len(X_test))

1166475
291619


In [ ]:
lengths = X_train["reconstructed_text"].str.split().str.len()

print("Max:", lengths.max())
print("Mean:", lengths.mean())
print("95th percentile:", lengths.quantile(0.95))
print("99th percentile:", lengths.quantile(0.99))

Max: 88
Mean: 13.136587582245655
95th percentile: 25.0
99th percentile: 30.0


In [6]:
!pip install transformers==4.49.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 157.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 120.4 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.23.0
    Uninstalling huggingface_hub-1.23.0:
      Successfully uninstalled huggingface_hub-1.23.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.13.1
    Uninstalling transformers-5.13.1:
      Successfully uninstalled transformers-5.13.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of t

In [7]:
import numpy as np
import tensorflow as tf

from transformers import (
    AutoTokenizer,
    TFAutoModelForSequenceClassification
)

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report

In [8]:
y_train_stance = np.load(f"{save_dir}/y_train_stance.npy")
y_test_stance = np.load(f"{save_dir}/y_test_stance.npy")

In [ ]:
sample_size = 100000

X_train_sample, _, y_train_sample, _ = train_test_split(
    X_train,
    y_train_stance,
    train_size=sample_size,
    stratify=y_train_stance,
    random_state=42
)

In [ ]:
X_train_text = X_train_sample["reconstructed_text"].tolist()
X_test_text = X_test["reconstructed_text"].tolist()

In [9]:
MODEL_NAME = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
MAX_LEN = 32

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
train_encodings = tokenizer(
    X_train_text,
    truncation=True,
    padding="max_length",
    max_length=MAX_LEN
)

test_encodings = tokenizer(
    X_test_text,
    truncation=True,
    padding="max_length",
    max_length=MAX_LEN
)

In [ ]:
train_encodings = {k: np.array(v) for k, v in train_encodings.items()}
test_encodings = {k: np.array(v) for k, v in test_encodings.items()}

In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np

all_train_encodings = train_encodings


train_idx, val_idx = train_test_split(
    np.arange(len(y_train_sample)),
    test_size=0.1,
    stratify=y_train_sample,
    random_state=42
)


In [ ]:
print("train_encodings:", len(train_encodings["input_ids"]))
print("all_train_encodings:", len(all_train_encodings["input_ids"]))
print("max train_idx:", train_idx.max())
print("max val_idx:", val_idx.max())

train_encodings: 100000
all_train_encodings: 100000
max train_idx: 99999
max val_idx: 99994


In [ ]:
train_encodings = {
    k: v[train_idx]
    for k, v in all_train_encodings.items()
}

val_encodings = {
    k: v[val_idx]
    for k, v in all_train_encodings.items()
}

y_train = y_train_sample[train_idx]
y_val = y_train_sample[val_idx]


In [ ]:
train_dataset = tf.data.Dataset.from_tensor_slices((
    train_encodings,
    y_train
))

val_dataset = tf.data.Dataset.from_tensor_slices((
    val_encodings,
    y_val
))

test_dataset = tf.data.Dataset.from_tensor_slices((
    test_encodings,
    y_test_stance
))

In [ ]:
BATCH_SIZE = 16

train_dataset = (
    train_dataset
    .shuffle(10000)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

val_dataset = (
    val_dataset
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

test_dataset = (
    test_dataset
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

In [ ]:
print(len(train_encodings["input_ids"]))
print(len(val_encodings["input_ids"]))
print(len(y_train_sample))
print(len(y_val))

90000
10000
100000
10000


In [ ]:
print(len(train_encodings["input_ids"]))
print(len(val_encodings["input_ids"]))
print(len(y_train))
print(len(y_val))

90000
10000
90000
10000


In [ ]:
print(len(y_train))
print(len(y_val))

print(tf.data.experimental.cardinality(train_dataset).numpy())
print(tf.data.experimental.cardinality(val_dataset).numpy())

90000
10000
5625
625


In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True,
    verbose=1
)

In [ ]:
weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train_sample),
    y=y_train_sample
)

class_weights = {
    0: weights[0],
    1: weights[1],
    2: weights[2]
}

print(class_weights)

{0: np.float64(19.8294665873488), 1: np.float64(4.280089025851738), 2: np.float64(0.3681980021576403)}


In [10]:
model = TFAutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3
)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

All PyTorch model weights were used when initializing TFBertForSequenceClassification.

Some weights or buffers of the TF 2.0 model TFBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
import tensorflow as tf
import transformers
import keras

print("TensorFlow:", tf.__version__)
print("Transformers:", transformers.__version__)
print("Keras:", keras.__version__)

TensorFlow: 2.20.0
Transformers: 4.49.0
Keras: 3.13.2


In [ ]:
optimizer = tf.keras.optimizers.Adam(
    learning_rate=2e-5
)

loss = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=True
)

In [ ]:
model.compile(
    optimizer=optimizer,
    loss=loss,
    metrics=["accuracy"]
)

In [ ]:
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=10,
    callbacks=[early_stopping]
)

Epoch 1/10
5625/5625 [==============================] - 903s 152ms/step - loss: 0.3016 - accuracy: 0.9087 - val_loss: 0.2864 - val_accuracy: 0.9073
Epoch 2/10
5625/5625 [==============================] - 874s 155ms/step - loss: 0.2534 - accuracy: 0.9188 - val_loss: 0.2859 - val_accuracy: 0.9102
Epoch 3/10
5625/5625 [==============================] - 846s 150ms/step - loss: 0.1955 - accuracy: 0.9332 - val_loss: 0.3348 - val_accuracy: 0.9031
Epoch 4/10
5625/5625 [==============================] - 834s 148ms/step - loss: 0.1280 - accuracy: 0.9554 - val_loss: 0.3827 - val_accuracy: 0.9019
Epoch 4: early stopping
Restoring model weights from the end of the best epoch: 2.


In [ ]:
preds = model.predict(test_dataset)

y_pred = np.argmax(
    preds.logits,
    axis=1
)

18227/18227 [==============================] - 1024s 56ms/step


In [ ]:
print(
    classification_report(
        y_test_stance,
        y_pred,
        target_names=[
            "Pro Russia",
            "Pro Ukraine",
            "Unsure"
        ]
    )
)

              precision    recall  f1-score   support

  Pro Russia       0.44      0.02      0.03      4902
 Pro Ukraine       0.54      0.37      0.44     22711
      Unsure       0.93      0.97      0.95    264006

    accuracy                           0.91    291619
   macro avg       0.64      0.45      0.47    291619
weighted avg       0.89      0.91      0.90    291619



In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test_stance, y_pred)

print(cm)

[[    78    262   4562]
 [    21   8369  14321]
 [    79   6992 256935]]


##model 2


In [ ]:
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=10,
    class_weight=class_weights,
    callbacks=[early_stopping]
)

Epoch 1/10
5625/5625 [==============================] - 876s 147ms/step - loss: 1.0200 - accuracy: 0.4922 - val_loss: 1.0051 - val_accuracy: 0.4802
Epoch 2/10
5625/5625 [==============================] - 815s 145ms/step - loss: 0.8810 - accuracy: 0.5839 - val_loss: 0.9160 - val_accuracy: 0.5482
Epoch 3/10
5625/5625 [==============================] - 817s 145ms/step - loss: 0.7409 - accuracy: 0.6352 - val_loss: 0.6072 - val_accuracy: 0.7216
Epoch 4/10
5625/5625 [==============================] - 815s 145ms/step - loss: 0.5920 - accuracy: 0.7007 - val_loss: 0.6201 - val_accuracy: 0.7192
Epoch 5/10
5625/5625 [==============================] - 811s 144ms/step - loss: 0.4616 - accuracy: 0.7487 - val_loss: 0.5147 - val_accuracy: 0.7833
Epoch 6/10
5625/5625 [==============================] - 809s 144ms/step - loss: 0.3615 - accuracy: 0.7856 - val_loss: 0.6019 - val_accuracy: 0.7522
Epoch 7/10
5625/5625 [==============================] - 813s 145ms/step - loss: 0.2988 - accuracy: 0.8179 - val_

In [ ]:
preds = model.predict(test_dataset)

y_pred = np.argmax(
    preds.logits,
    axis=1
)

18227/18227 [==============================] - 1004s 55ms/step


In [ ]:
print(
    classification_report(
        y_test_stance,
        y_pred,
        target_names=[
            "Pro Russia",
            "Pro Ukraine",
            "Unsure"
        ]
    )
)

              precision    recall  f1-score   support

  Pro Russia       0.09      0.17      0.12      4902
 Pro Ukraine       0.25      0.63      0.36     22711
      Unsure       0.95      0.82      0.88    264006

    accuracy                           0.79    291619
   macro avg       0.43      0.54      0.45    291619
weighted avg       0.88      0.79      0.83    291619



In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test_stance, y_pred)

print(cm)

[[   844   1353   2705]
 [   724  14268   7719]
 [  7999  40735 215272]]


##model 3

In [ ]:
class_weights = {
    0: 10,
    1: 3,
    2: 1
}

In [ ]:
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=10,
    class_weight=class_weights,
    callbacks=[early_stopping]
)

Epoch 1/10
5625/5625 [==============================] - 833s 146ms/step - loss: 0.4069 - accuracy: 0.8943 - val_loss: 0.4105 - val_accuracy: 0.8680
Epoch 2/10
5625/5625 [==============================] - 815s 145ms/step - loss: 0.3169 - accuracy: 0.9182 - val_loss: 0.4622 - val_accuracy: 0.8505
Epoch 3/10
5625/5625 [==============================] - 816s 145ms/step - loss: 0.2468 - accuracy: 0.9377 - val_loss: 0.4852 - val_accuracy: 0.8741
Epoch 3: early stopping
Restoring model weights from the end of the best epoch: 1.


In [ ]:
preds = model.predict(test_dataset)

y_pred = np.argmax(
    preds.logits,
    axis=1
)

18227/18227 [==============================] - 985s 54ms/step


In [ ]:
print(
    classification_report(
        y_test_stance,
        y_pred,
        target_names=[
            "Pro Russia",
            "Pro Ukraine",
            "Unsure"
        ]
    )
)

              precision    recall  f1-score   support

  Pro Russia       0.14      0.08      0.10      4902
 Pro Ukraine       0.35      0.46      0.40     22711
      Unsure       0.94      0.92      0.93    264006

    accuracy                           0.87    291619
   macro avg       0.47      0.49      0.48    291619
weighted avg       0.88      0.87      0.87    291619



In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test_stance, y_pred)

print(cm)

[[   388    632   3882]
 [   263  10507  11941]
 [  2209  19086 242711]]


##model 4

In [ ]:
class_weights = {
    0: 15,
    1: 4,
    2: 1
}

In [ ]:
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=10,
    class_weight=class_weights,
    callbacks=[early_stopping]
)

Epoch 1/10
5625/5625 [==============================] - 910s 153ms/step - loss: 1.1753 - accuracy: 0.8629 - val_loss: 0.5520 - val_accuracy: 0.8069
Epoch 2/10
5625/5625 [==============================] - 844s 150ms/step - loss: 0.9716 - accuracy: 0.8434 - val_loss: 0.4345 - val_accuracy: 0.8436
Epoch 3/10
5625/5625 [==============================] - 840s 149ms/step - loss: 0.7754 - accuracy: 0.8524 - val_loss: 0.4256 - val_accuracy: 0.8239
Epoch 4/10
5625/5625 [==============================] - 843s 150ms/step - loss: 0.5561 - accuracy: 0.8847 - val_loss: 0.3743 - val_accuracy: 0.8755
Epoch 5/10
5625/5625 [==============================] - 833s 148ms/step - loss: 0.4099 - accuracy: 0.9101 - val_loss: 0.3929 - val_accuracy: 0.8736
Epoch 6/10
5625/5625 [==============================] - 829s 147ms/step - loss: 0.3043 - accuracy: 0.9321 - val_loss: 0.4754 - val_accuracy: 0.8577
Epoch 6: early stopping
Restoring model weights from the end of the best epoch: 4.


In [ ]:
preds = model.predict(test_dataset)

y_pred = np.argmax(
    preds.logits,
    axis=1
)

18227/18227 [==============================] - 1016s 56ms/step


In [ ]:
print(
    classification_report(
        y_test_stance,
        y_pred,
        target_names=[
            "Pro Russia",
            "Pro Ukraine",
            "Unsure"
        ]
    )
)

              precision    recall  f1-score   support

  Pro Russia       0.11      0.16      0.13      4902
 Pro Ukraine       0.42      0.45      0.43     22711
      Unsure       0.94      0.93      0.93    264006

    accuracy                           0.88    291619
   macro avg       0.49      0.51      0.50    291619
weighted avg       0.89      0.88      0.88    291619



In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test_stance, y_pred)

print(cm)

[[   771    458   3673]
 [   595  10320  11796]
 [  5748  14015 244243]]


##train for entire data


In [11]:
from sklearn.model_selection import train_test_split
import tensorflow as tf

X_train_final, X_val, y_train_final, y_val = train_test_split(
    X_train,
    y_train_stance,
    test_size=0.1,
    stratify=y_train_stance,
    random_state=42
)

In [12]:
X_train_text = X_train_final["reconstructed_text"].tolist()
X_val_text = X_val["reconstructed_text"].tolist()
X_test_text = X_test["reconstructed_text"].tolist()

In [13]:
MAX_LEN = 32

train_encodings = tokenizer(
    X_train_text,
    truncation=True,
    padding="max_length",
    max_length=MAX_LEN,
    return_tensors="tf"
)

val_encodings = tokenizer(
    X_val_text,
    truncation=True,
    padding="max_length",
    max_length=MAX_LEN,
    return_tensors="tf"
)

test_encodings = tokenizer(
    X_test_text,
    truncation=True,
    padding="max_length",
    max_length=MAX_LEN,
    return_tensors="tf"
)

In [14]:
train_dataset = tf.data.Dataset.from_tensor_slices((
    dict(train_encodings),
    y_train_final
))

val_dataset = tf.data.Dataset.from_tensor_slices((
    dict(val_encodings),
    y_val
))

test_dataset = tf.data.Dataset.from_tensor_slices((
    dict(test_encodings),
    y_test_stance
))

In [15]:
BATCH_SIZE = 16

train_dataset = (
    train_dataset
    .shuffle(10000)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

val_dataset = (
    val_dataset
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

test_dataset = (
    test_dataset
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

In [16]:
import os
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    CSVLogger
)

SAVE_DIR_1 = "/content/drive/MyDrive/New_BERT_Stance_Final"
os.makedirs(SAVE_DIR_1, exist_ok=True)


epoch_checkpoint = ModelCheckpoint(
    filepath=os.path.join(
        SAVE_DIR_1,
        "checkpoint_epoch_{epoch:02d}"
    ),
    monitor="val_loss",
    save_best_only=False,
    save_weights_only=False,
    save_freq="epoch",
    verbose=1
)

best_checkpoint = ModelCheckpoint(
    filepath=os.path.join(
        SAVE_DIR_1,
        "best_model"
    ),
    monitor="val_loss",
    mode="min",
    save_best_only=True,
    save_weights_only=False,
    verbose=1
)

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True,
    verbose=1
)

csv_logger = CSVLogger(
    os.path.join(SAVE_DIR_1, "training_log.csv"),
    append=True
)

In [17]:
optimizer = tf.keras.optimizers.Adam(
    learning_rate=2e-5
)

loss = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=True
)

In [18]:
model.compile(
    optimizer=optimizer,
    loss=loss,
    metrics=["accuracy"]
)

In [19]:
class_weights = {
    0: 15,
    1: 4,
    2: 1
}

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=10,
    class_weight=class_weights,
    callbacks=[
        epoch_checkpoint,
        best_checkpoint,
        early_stopping,
        csv_logger
    ]
)

Epoch 1/10
65615/65615 [==============================] - ETA: 0s - loss: 1.0164 - accuracy: 0.8581
Epoch 1: saving model to /content/drive/MyDrive/New_BERT_Stance_Final/checkpoint_epoch_01

Epoch 1: val_loss improved from inf to 0.42204, saving model to /content/drive/MyDrive/New_BERT_Stance_Final/best_model
65615/65615 [==============================] - 4154s 63ms/step - loss: 1.0164 - accuracy: 0.8581 - val_loss: 0.4220 - val_accuracy: 0.8421
Epoch 2/10
65615/65615 [==============================] - ETA: 0s - loss: 0.8984 - accuracy: 0.8534
Epoch 2: saving model to /content/drive/MyDrive/New_BERT_Stance_Final/checkpoint_epoch_02

Epoch 2: val_loss improved from 0.42204 to 0.39810, saving model to /content/drive/MyDrive/New_BERT_Stance_Final/best_model
65615/65615 [==============================] - 4122s 63ms/step - loss: 0.8984 - accuracy: 0.8534 - val_loss: 0.3981 - val_accuracy: 0.8588
Epoch 3/10
65615/65615 [==============================] - ETA: 0s - loss: 0.8196 - accuracy: 0.8

In [20]:
preds = model.predict(test_dataset)

y_pred = np.argmax(
    preds.logits,
    axis=1
)

18227/18227 [==============================] - 398s 22ms/step


In [21]:
print(
    classification_report(
        y_test_stance,
        y_pred,
        target_names=[
            "Pro Russia",
            "Pro Ukraine",
            "Unsure"
        ]
    )
)

              precision    recall  f1-score   support

  Pro Russia       0.17      0.21      0.19      4902
 Pro Ukraine       0.39      0.60      0.48     22711
      Unsure       0.95      0.91      0.93    264006

    accuracy                           0.87    291619
   macro avg       0.51      0.57      0.53    291619
weighted avg       0.90      0.87      0.88    291619



In [22]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test_stance, y_pred)

print(cm)

[[  1044    584   3274]
 [   488  13694   8529]
 [  4631  20392 238983]]
